# Prueba Técnica - Limpieza e Integración de Datos

En este notebook se realiza el proceso de exploración, limpieza, transformación e integración de los archivos 'transacciones.csv' y 'clientes.json', utilizando PySpark.

# Importación de librerías

In [1]:
import os
import sys
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    initcap,
    when,
    regexp_replace,
    to_date,
    coalesce,
    lit,
    length,
    translate,
    count,
    isnan,
    abs,
    expr,
    concat_ws
)

from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    StringType,
    DateType
)

# Inicio Sesión PySpark

In [2]:
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Prueba")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .getOrCreate()
)

print("Python:", spark.sparkContext.pythonExec)
print("Spark:", spark.version)

Python: c:\Users\user\Documents\prueba_tecnica_analista_junior_iris\.venv\Scripts\python.exe
Spark: 3.5.6


# Carga de Data

In [4]:
clientes = (
    spark.read
    .option("multiline", True)
    .json("../data/clientes.json")
)

transacciones = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv("../data/transacciones.csv")
)

# Exploración de la Data

In [5]:
clientes.printSchema()
clientes.show(5, truncate=False)

transacciones.printSchema()
transacciones.show(5, truncate=False)

root
 |-- activo: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- contacto: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- telefono: string (nullable = true)
 |-- fecha_alta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- productos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- segmento: string (nullable = true)
 |-- tipo_documento: string (nullable = true)

+------+---------+--------------------------------------------+-----------+----------+------------------+----------------+----------------------------------+--------+--------------+
|activo|ciudad   |contacto                                    |fecha_alta |id_cliente|nombre            |numero_documento|productos                         |segmento|tipo_documento|
+------+---------+--------------------------------------------+-----------+--------

In [6]:
transacciones.printSchema()
transacciones.show(5, truncate=False)

root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_producto: string (nullable = true)
 |-- monto: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tasa_interes: string (nullable = true)
 |-- plazo_dias: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+----------+-------------+-------------+-------+------------+----------+-----+--------+------------+
|id_transaccion|fecha     |id_cliente|tipo_producto|monto        |moneda |tasa_interes|plazo_dias|canal|estado  |sucursal    |
+--------------+----------+----------+-------------+-------------+-------+------------+----------+-----+--------+------------+
|500058        |2025-03-04|1013      |CTA_AHORROS  |1.236.028 COP|Pesos  |NULL        |0         |Web  |pend    |Barranquilla|
|500240        |2024-02-16|1007      |cd

In [7]:
transacciones.select("estado").distinct().show(100, False)
transacciones.select("canal").distinct().show(100, False)

+----------+
|estado    |
+----------+
|APROBADA  |
|pend      |
|Rechazada |
|rechazada |
|Pendiente |
|Aprobada  |
|aprobado  |
|En proceso|
|PENDIENTE |
+----------+

+-----------------+
|canal            |
+-----------------+
|APP              |
|oficina          |
|ATM              |
|Cajero           |
|app movil        |
|App              |
|OFICINA          |
|web              |
|cajero automatico|
|Web              |
|Oficina          |
+-----------------+



In [8]:
transacciones.select("moneda").distinct().show(100, False)
transacciones.select("sucursal").distinct().show(100, False)

+-------+
|moneda |
+-------+
|Dolares|
|usd    |
|COP    |
|pesos  |
|cop    |
|Pesos  |
|$      |
|USD    |
|NULL   |
+-------+

+------------+
|sucursal    |
+------------+
|cali        |
|Bucaramanga |
|b/quilla    |
|  Bogotá    |
|medellín    |
|Cali        |
|BOGOTA      |
|Medellin    |
|Bogota      |
|Cartagena   |
|Barranquilla|
|NULL        |
+------------+



In [9]:
transacciones.select("tipo_producto").distinct().show(100, False)
clientes.select("segmento").distinct().show(100, False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|ahorros                |
|CDT                    |
|CORRIENTE              |
|CTA_AHORROS            |
|cdt                    |
|Crédito                |
|Credito                |
|crédito                |
|CERTIFICADO DE DEPOSITO|
|Certificado de Deposito|
|Cuenta Ahorros         |
|cta corriente          |
|Cuenta corriente       |
|AHORROS                |
|CREDITO                |
|Cdt                    |
|Cuenta Corriente       |
|Cuenta de ahorros      |
+-----------------------+

+-----------+
|segmento   |
+-----------+
|Premium    |
|Personal   |
|pyme       |
|Empresarial|
|Pyme       |
|EMPRESARIAL|
|personal   |
|premium    |
|NULL       |
+-----------+



In [10]:
clientes.select("tipo_documento").distinct().show(100, False)
clientes.select("activo").distinct().show(100, False)

+--------------+
|tipo_documento|
+--------------+
|cc            |
|nit           |
|NIT           |
|CC            |
|C.C.          |
|NULL          |
+--------------+

+------+
|activo|
+------+
|0     |
|No    |
|false |
|Si    |
|1     |
|SI    |
|NO    |
|true  |
+------+



# Exploración de Duplicados

In [11]:
transacciones.groupBy("id_transaccion") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+--------------+-----+
|id_transaccion|count|
+--------------+-----+
|        500148|    2|
|        500363|    2|
|        500153|    2|
|        500060|    2|
|        500123|    2|
|        500260|    2|
+--------------+-----+



In [12]:
transacciones.filter(
    col("id_transaccion").isin(
        "500148",
        "500363",
        "500153",
        "500060",
        "500123",
        "500260"
    )
).orderBy("id_transaccion").show(truncate=False)

+--------------+----------+----------+-----------------------+-------------+------+------------+----------+---------+----------+-----------+
|id_transaccion|fecha     |id_cliente|tipo_producto          |monto        |moneda|tasa_interes|plazo_dias|canal    |estado    |sucursal   |
+--------------+----------+----------+-----------------------+-------------+------+------------+----------+---------+----------+-----------+
|500060        |25.10.2024| 1026     |Cuenta de ahorros      |7.784.912,25 |NULL  |0           |0         |oficina  |En proceso|Bucaramanga|
|500060        |25.10.2024| 1026     |Cuenta de ahorros      |7.784.912,25 |NULL  |0           |0         |oficina  |En proceso|Bucaramanga|
|500123        |20-11-2024|7777      |ahorros                |1.973.695,25 |Pesos |N/A         |NULL      |app movil|Rechazada |b/quilla   |
|500123        |20-11-2024|7777      |ahorros                |1.973.695,25 |Pesos |N/A         |NULL      |app movil|Rechazada |b/quilla   |
|500148      

In [13]:
clientes.groupBy("id_cliente") \
    .count() \
    .filter(col("count") > 1) \
    .show(20, False)

+----------+-----+
|id_cliente|count|
+----------+-----+
|1004      |2    |
|1011      |2    |
+----------+-----+



In [14]:
clientes.filter(col("id_cliente") == "1004").show(20, False)

+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+
|activo|ciudad  |contacto                                  |fecha_alta|id_cliente|nombre       |numero_documento|productos       |segmento|tipo_documento|
+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+
|true  |medellín|{isabella.diaz@empresa.com.co, 3138849654}|7-1-2022  |1004      |Isabella Diaz|342601915       |[Cuenta Ahorros]|premium |C.C.          |
|true  |medellín|{isabella.diaz@empresa.com.co, 3138849654}|7-1-2022  |1004      |Isabella Diaz|342601915       |[Cuenta Ahorros]|premium |C.C.          |
+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+



In [15]:
clientes.filter(col("id_cliente") == "1011").show(20, False)

+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+
|activo|ciudad|contacto              |fecha_alta|id_cliente|nombre         |numero_documento|productos         |segmento|tipo_documento|
+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+
|false |cali  |{NULL, (602) 560 7689}|2022-11-27|1011      |SANTIAGO VARGAS|339147326       |[Cuenta Corriente]|premium |NIT           |
|false |cali  |{NULL, (602) 560 7689}|2022-11-27|1011      |SANTIAGO VARGAS|339147326       |[Cuenta Corriente]|premium |NIT           |
+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+



# Eliminación de Duplicados

In [16]:
print("Cantidad antes:", transacciones.count())
transacciones = transacciones.dropDuplicates()
print("Cantidad después:", transacciones.count())

Cantidad antes: 429
Cantidad después: 423


In [17]:
clientes = clientes.dropDuplicates()
clientes.groupBy("id_cliente") \
    .count() \
    .filter(col("count") > 1) \
    .show(20, False)


+----------+-----+
|id_cliente|count|
+----------+-----+
+----------+-----+



# Eliminar Espacios

In [18]:
for columna in transacciones.columns:
    transacciones = transacciones.withColumn(
        columna,
        trim(col(columna))
    )

# Exploración de Valores Vacios

In [19]:
for columna in transacciones.columns:
    cantidad = transacciones.filter(col(columna) == "").count()
    print(f"{columna}: {cantidad}")

id_transaccion: 0
fecha: 0
id_cliente: 0
tipo_producto: 0
monto: 0
moneda: 0
tasa_interes: 0
plazo_dias: 0
canal: 0
estado: 0
sucursal: 0


# Limpieza Transacciones

# Unificar texto en columna tipo_producto

In [20]:
transacciones = transacciones.replace(
    {
        "CERTIFICADO DE DEPOSITO": "CDT",
        "CTA_AHORROS": "CUENTA DE AHORROS",
        "AHORROS": "CUENTA DE AHORROS",
        "CUENTA AHORROS": "CUENTA DE AHORROS",
        "CTA CORRIENTE": "CUENTA CORRIENTE",
        "CORRIENTE": "CUENTA CORRIENTE",
        "CREDITO": "CRÉDITO"
    },
    subset=["tipo_producto"]
)

In [21]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    initcap(col("tipo_producto"))
)

In [22]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    regexp_replace(col("tipo_producto"), " De ", " de ")
)

In [23]:
transacciones.select("tipo_producto").distinct().show(truncate=False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|Ahorros                |
|Crédito                |
|Credito                |
|Certificado de Deposito|
|Cuenta Ahorros         |
|Cuenta de Ahorros      |
|Cdt                    |
|Cuenta Corriente       |
|Cta Corriente          |
+-----------------------+



# Convertir Cdt --> CDT

In [24]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    when(col("tipo_producto") == "Cdt", "CDT")
    .otherwise(col("tipo_producto"))
)

In [25]:
transacciones.select("tipo_producto").distinct().show(truncate=False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|Ahorros                |
|CDT                    |
|Crédito                |
|Credito                |
|Certificado de Deposito|
|Cuenta Ahorros         |
|Cuenta de Ahorros      |
|Cuenta Corriente       |
|Cta Corriente          |
+-----------------------+



# Limpieza de la columna estado

# Unificar columna estado

In [26]:
transacciones = transacciones.withColumn(
    "estado",
    upper(col("estado"))
)

# Estandarizar columna estado

In [27]:
transacciones = transacciones.replace(
    {
        "APROBADO": "APROBADA",
        "PEND": "PENDIENTE",
        "EN PROCESO": "PENDIENTE"
    },
    subset=["estado"]
)

In [28]:
transacciones = transacciones.withColumn(
    "estado",
    initcap(col("estado"))
)

In [29]:
transacciones.select("estado").distinct().show()

+---------+
|   estado|
+---------+
|Rechazada|
|Pendiente|
| Aprobada|
+---------+



# Limpieza columna canal

# Unificar columna canal

In [30]:
transacciones = transacciones.withColumn(
    "canal",
    upper(col("canal"))
)

# Estandarizar columna canal

In [31]:
transacciones = transacciones.replace(
    {
        "APP MOVIL": "APP",
        "ATM": "CAJERO",
        "CAJERO AUTOMATICO": "CAJERO"
    },
    subset=["canal"]
)

In [32]:
transacciones = transacciones.withColumn(
    "canal",
    initcap(col("canal"))
)

In [33]:
transacciones.select("canal").distinct().show()

+-------+
|  canal|
+-------+
| Cajero|
|    App|
|    Web|
|Oficina|
+-------+



# Limpieza de la columna moneda

# Unificar columna moneda

In [34]:
transacciones = transacciones.withColumn(
    "moneda",
    upper(col("moneda"))
)

# Estandarizar columna moneda

In [35]:
transacciones = transacciones.replace(
    {
        "PESOS": "COP",
        "$": "COP",
        "DOLARES": "USD"
    },
    subset=["moneda"]
)

In [36]:
transacciones.select("moneda").distinct().show()

+------+
|moneda|
+------+
|   COP|
|   USD|
|  NULL|
+------+

